# 🚀 FaceKey Animation Studio — Kaggle Data Collector & GPU Trainer
### Parallel Multi-Cloud Execution (Concurrent with Google Colab)

**Why Kaggle?**
- **30 Hours of Free GPU Weekly** (NVIDIA P100 16GB or 2x NVIDIA T4 16GB).
- **Runs Simultaneously with Google Colab**: Colab and Kaggle process different batches in parallel, doubling your data collection and training throughput!
- **Distributed Locking**: No redundant downloads! Atomic locks in Google Drive ensure Colab and Kaggle never process the same video.
- **Session Expiry Recovery**: If a Kaggle 12-hour session ends or is preempted, locks are automatically released or reclaimed on next start.

## Step 1: Verify Kaggle GPU Accelerator & Install Dependencies (Self-Healing)
In the Kaggle Notebook sidebar: Ensure **Accelerator** is set to **GPU P100** or **GPU T4 x2**, and **Internet** is set to **Internet On**.

In [ ]:
!nvidia-smi

import os, sys, shutil

# Always reset working directory to root /kaggle/working first
%cd /kaggle/working

REPO_URL = 'https://github.com/Vijay-1010110/FaceKeyAnimation.git'
APP_DIR = '/kaggle/working/FaceKeyAnimation'

# Self-healing repository clone/reset
if os.path.isdir(os.path.join(APP_DIR, '.git')):
    print('[*] Updating existing repository to latest origin/main...')
    !cd {APP_DIR} && git fetch --all --prune && git reset --hard origin/main && git clean -fd
else:
    print('[*] Performing fresh clone of FaceKeyAnimation...')
    !rm -rf {APP_DIR}
    !git clone {REPO_URL} {APP_DIR}

%cd {APP_DIR}

# Install dependencies
!pip install -q yt-dlp mediapipe opencv-python-headless numpy sounddevice mss av torch torchvision torchaudio
print('[+] Kaggle environment and repository ready!')

## Step 2: Configure Google Drive / Cloud Storage on Kaggle
Kaggle allows connecting Google Drive through multiple methods:
1. **Add Storage → Google Drive**: In the right sidebar, click **+ Add Input**, select **Google Drive**, and authenticate. Drive mounts under `/kaggle/input/google-drive`.
2. **Custom / Working Directory**: If running standalone on Kaggle, data will be organized in `/kaggle/working/FaceKeyDataset`.

In [ ]:
# Detect Drive location on Kaggle
possible_drive_paths = [
    '/kaggle/input/google-drive/FaceKeyDataset',
    '/kaggle/input/facekeydataset/FaceKeyDataset',
    '/kaggle/working/google_drive/FaceKeyDataset',
    '/kaggle/working/FaceKeyDataset'
]

DRIVE_DIR = '/kaggle/working/FaceKeyDataset'
for p in possible_drive_paths:
    if os.path.exists(p):
        DRIVE_DIR = p
        break

os.makedirs(f'{DRIVE_DIR}/chunks', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/locks', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)

print(f'[+] Using Dataset Root: {DRIVE_DIR}')

## Step 3: Run Kaggle Batch Data Collector (Worker `kaggle-worker-1`)
Kaggle processes its dedicated queue `youtube_urls_kaggle.txt` (Categories: MIT OCW, Stanford Online, Closer to Truth, Lex Fridman, Numberphile):
- Operates in lock-step with Colab: checks shared registry on Drive to ensure zero redundancy.
- Heartbeat keeps lock active; if session expires after 12 hrs, locks can be safely reclaimed.

In [ ]:
urls_file = os.path.join(APP_DIR, 'sessions', 'youtube_urls_kaggle.txt')

!python scripts/cloud_data_collector.py \
    --drive-dir "{DRIVE_DIR}" \
    --worker-id "kaggle-worker-1" \
    --urls-file "{urls_file}" \
    --chunk-size 5 \
    --quality 480p

## Step 4: Run Fast GPU Model Training on Kaggle (P100 / Dual T4)
Unpack chunks to Kaggle's high-speed NVMe scratch disk and train the `SpeechToFaceAnimator` model with Automatic Mixed Precision (FP16).

In [ ]:
# Unpack chunks to fast local NVMe SSD
LOCAL_DATA_DIR = '/kaggle/working/training_data'

from src.storage.cloud_sync import CloudDriveSync
sync = CloudDriveSync(project_root=APP_DIR, drive_folder=DRIVE_DIR)
sync.unpack_all_chunks_to_local_ssd(LOCAL_DATA_DIR)

# Launch PyTorch Model Training
!python scripts/train_speech_to_animation.py \
    --drive-dir "{DRIVE_DIR}" \
    --batch-size 64 \
    --epochs 50 \
    --seq-len 64